# Entrenamiento v2 — TF-IDF + Features manuales

En este notebook mejoramos el modelo baseline añadiendo features manuales al TF-IDF.

**Baseline (notebook 3):** LogisticRegression + TF-IDF → F1-macro validación: 0.87

**Mejoras en este notebook:**
1. Features manuales: longitud de texto + conteo de keywords de dominio por clase
2. Concatenación TF-IDF + features manuales
3. Re-entrenamiento de LogisticRegression
4. Comparativa con baseline
5. Registro en MLflow (Experimento 1)

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Carga de datos

In [2]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/validation.csv")
test_df = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val = val_df["text_final"]
y_val = val_df["etiqueta"]
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

Train: 210 muestras
Validation: 45 muestras
Test: 45 muestras


## 2. Vectorización TF-IDF (misma config que baseline)

In [3]:
from functions import crear_tfidf

tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train, X_val, X_test,
    max_features=5000,
    ngram_range=(1, 2),
)

OSError: [WinError 1114] Error en una rutina de inicialización de biblioteca de vínculos dinámicos (DLL). Error loading "c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 3. Features manuales

In [ ]:
from functions import crear_features_manuales, combinar_features, KEYWORDS_DOMINIO

# Generar features manuales para cada conjunto
feat_train = crear_features_manuales(X_train)
feat_val = crear_features_manuales(X_val)
feat_test = crear_features_manuales(X_test)

print(f"Features manuales: {list(feat_train.columns)}")
print(f"Forma: {feat_train.shape}")
print()
print(feat_train.describe().round(2))

Features manuales: ['num_palabras', 'num_caracteres', 'kw_inaceptable', 'kw_alto_riesgo', 'kw_riesgo_limitado', 'kw_riesgo_minimo']
Forma: (210, 6)

       num_palabras  num_caracteres  kw_inaceptable  kw_alto_riesgo  \
count        210.00          210.00          210.00          210.00   
mean          24.94          228.48            0.22            0.29   
std            5.53           50.46            0.65            0.51   
min            4.00           28.00            0.00            0.00   
25%           22.00          198.00            0.00            0.00   
50%           25.00          224.50            0.00            0.00   
75%           27.75          250.00            0.00            1.00   
max           45.00          410.00            4.00            3.00   

       kw_riesgo_limitado  kw_riesgo_minimo  
count              210.00            210.00  
mean                 1.11              0.22  
std                  0.60              0.57  
min                  0.00  

In [ ]:
# Concatenar TF-IDF + features manuales
X_train_combined = combinar_features(X_train_tfidf, feat_train)
X_val_combined = combinar_features(X_val_tfidf, feat_val)
X_test_combined = combinar_features(X_test_tfidf, feat_test)

print(f"Forma train combinada: {X_train_combined.shape}")
print(f"Forma validation combinada: {X_val_combined.shape}")
print(f"Forma test combinada: {X_test_combined.shape}")
print(f"  (TF-IDF: {X_train_tfidf.shape[1]} + manuales: {feat_train.shape[1]} = {X_train_combined.shape[1]})")

Forma train combinada: (210, 3779)
Forma validation combinada: (45, 3779)
Forma test combinada: (45, 3779)
  (TF-IDF: 3773 + manuales: 6 = 3779)


## 4. Entrenamiento LogisticRegression con features combinadas

In [ ]:
from functions import entrenar_modelo_baseline

modelo_v2 = entrenar_modelo_baseline(X_train_combined, y_train, X_val_combined, y_val)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.33      0.14      0.20         7
    inaceptable       0.67      0.90      0.77        20
riesgo_limitado       0.00      0.00      0.00         3
  riesgo_minimo       0.87      0.87      0.87        15

       accuracy                           0.71        45
      macro avg       0.47      0.48      0.46        45
   weighted avg       0.64      0.71      0.66        45

F1-score macro (validación): 0.4582


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

## 5. Comparativa con baseline

In [ ]:
from sklearn.metrics import f1_score

# Métricas del baseline (notebook 3)
BASELINE_F1_MACRO = 0.8698
BASELINE_ACCURACY = 0.87

# Métricas del modelo v2
y_val_pred_v2 = modelo_v2.predict(X_val_combined)
f1_v2 = f1_score(y_val, y_val_pred_v2, average="macro")
acc_v2 = (y_val_pred_v2 == y_val).mean()

print("=== COMPARATIVA VALIDACIÓN ===")
print(f"{'Métrica':<20} {'Baseline':>10} {'V2 (+ features)':>16} {'Diferencia':>12}")
print("-" * 60)
print(f"{'F1-macro':<20} {BASELINE_F1_MACRO:>10.4f} {f1_v2:>16.4f} {f1_v2 - BASELINE_F1_MACRO:>+12.4f}")
print(f"{'Accuracy':<20} {BASELINE_ACCURACY:>10.4f} {acc_v2:>16.4f} {acc_v2 - BASELINE_ACCURACY:>+12.4f}")

=== COMPARATIVA VALIDACIÓN ===
Métrica                Baseline  V2 (+ features)   Diferencia
------------------------------------------------------------
F1-macro                 0.8698           0.4582      -0.4116
Accuracy                 0.8700           0.7111      -0.1589


## 6. Guardar artefactos

In [ ]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo_v2, os.path.join(output_dir, "modelo_logreg_v2.joblib"))
joblib.dump(modelo_v2, os.path.join(output_dir, "modelo_clasificador.joblib"))
joblib.dump(tfidf, os.path.join(output_dir, "tfidf_vectorizer.joblib"))

print("Modelo v2 guardado en: model/modelo_logreg_v2.joblib")
print("Modelo v2 guardado en: model/modelo_clasificador.joblib")
print("Vectorizador guardado en: model/tfidf_vectorizer.joblib")

Modelo v2 guardado en: model/modelo_logreg_v2.joblib
Modelo v2 guardado en: model/modelo_clasificador.joblib
Vectorizador guardado en: model/tfidf_vectorizer.joblib


## 7. Registro en MLflow — Experimento 1

In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="v2_tfidf_features_manuales"):
        mlflow.log_param("tfidf_max_features",     5000)
        mlflow.log_param("tfidf_ngram_range",      "(1, 2)")
        mlflow.log_param("tfidf_sublinear_tf",     True)
        mlflow.log_param("modelo",                 "LogisticRegression")
        mlflow.log_param("max_iter",               1000)
        mlflow.log_param("features_manuales",      list(feat_train.columns))
        mlflow.log_param("n_keywords_por_clase",   len(list(KEYWORDS_DOMINIO.values())[0]))
        mlflow.log_param("total_features",         X_train_combined.shape[1])

        mlflow.log_metric("val_f1_macro",          f1_v2)
        mlflow.log_metric("val_accuracy",          acc_v2)
        mlflow.log_metric("baseline_f1_macro",     BASELINE_F1_MACRO)

        print("✓ Exp 1 registrado en MLflow")
        print(f"  Val F1-macro: {f1_v2:.4f}")
        print(f"  Run ID: {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://34.244.146.100
⚠ MLflow no disponible: API request to https://34.244.146.100/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPSConnectionPool(host='34.244.146.100', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_ia_artificial (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)')))
